# Phase 1 — dense RAG baseline

This notebook follows the Phase 1 plan in `docs/learning-plan.md`.

## Step 0 — Environment and tiny corpus (this section)

- **Stack:** Python 3.11+, Jupyter, NumPy (cosine later), **one** embedding model via [sentence-transformers](https://www.sentence-transformers.org/) (local, consistent for Phase 1).
- **Corpus:** `data/phase1/*.md` — short docs with synonyms, near-duplicates, and mixed topics.
- **Done when:** you can run the corpus listing cell and see all files; venv has `requirements-phase1.txt` installed.

### 0.1 — Verify Python version

Target: **3.11+** (per learning plan).

In [1]:
import sys

vi = sys.version_info
print(f"Python {vi.major}.{vi.minor}.{vi.micro}")
assert vi.major == 3 and vi.minor >= 11, "Use Python 3.11 or newer for Phase 1"

Python 3.11.11


### 0.2 — List corpus files (`data/phase1`)

Run this from the **repository root**, or adjust `REPO_ROOT` below if you launched Jupyter from another cwd.

In [2]:
from pathlib import Path

# If necessary, set this to the meridian-context repo root:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data" / "phase1").is_dir():
    REPO_ROOT = REPO_ROOT.parent

corpus_dir = REPO_ROOT / "data" / "phase1"
assert corpus_dir.is_dir(), f"Missing corpus dir: {corpus_dir.resolve()}"

corpus_files = sorted(corpus_dir.glob("*.md"))
print(f"Corpus directory: {corpus_dir.resolve()}\n")
for i, p in enumerate(corpus_files, start=1):
    print(f"{i:2}. {p.name} ({p.stat().st_size} bytes)")

print(f"\nTotal: {len(corpus_files)} markdown files")

Corpus directory: /Users/naveed/LEARNING/AI/meridian-context/data/phase1

 1. 01_embeddings_intro.md (552 bytes)
 2. 02_embeddings_intro_paraphrase.md (519 bytes)
 3. 03_ann_hnsw_sketch.md (5692 bytes)
 4. 04_rag_vs_long_context.md (654 bytes)
 5. 05_vehicle_lexicon_automobile.md (325 bytes)
 6. 06_vehicle_lexicon_car.md (352 bytes)
 7. 07_recipe_marinara.md (262 bytes)
 8. 08_recipe_marinara_near_dup.md (310 bytes)
 9. README.md (482 bytes)

Total: 9 markdown files


### 0.3 — Stack sanity check (imports)

After `pip install -r requirements-phase1.txt`, this cell should run without `ImportError`.

**Embedding model (locked for Phase 1):** `all-MiniLM-L6-v2` — small, fast, good for exercises; first run downloads weights.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBED_MODEL_NAME)

probe = model.encode(["sanity check sentence"], convert_to_numpy=True)
print(f"Model: {EMBED_MODEL_NAME}")
print(f"Embedding shape: {probe.shape}  (batch, dims)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: all-MiniLM-L6-v2
Embedding shape: (1, 384)  (batch, dims)


### Next (Step 1+)

- Embeddings concept + micro-practice (single sentence, shape, optional paraphrase pairs).
- Cosine similarity + brute-force top-k over chunks (Step 2).
- See `docs/learning-plan.md` and your Phase 1 execution plan for checkpoints.